In [2]:
import pandas as pd
import random
import os

# Define paths
PROCESSED_DIR = r"C:\Users\HP\Desktop\thesis_preprocessing\data\processed"
INPUT_FILE = os.path.join(PROCESSED_DIR, "MASTER_original_12k.csv")

# Load data and remove nulls
df = pd.read_csv(INPUT_FILE)
df = df.dropna(subset=['original_comment']).reset_index(drop=True)
df['word_count'] = df['original_comment'].apply(lambda x: len(str(x).split()))

# 1. Word Jaccard (For Distractor A)
def word_jaccard(str1, str2):
    set1 = set(str1.lower().split())
    set2 = set(str2.lower().split())
    if not set1 or not set2:
        return 0.0
    return len(set1.intersection(set2)) / len(set1.union(set2))

# 2. Bigram Jaccard (For Distractor B)
def get_bigrams(words):
    return set(zip(words[:-1], words[1:]))

def bigram_jaccard(words1, words2):
    if len(words1) < 2 or len(words2) < 2:
        return 0.0
    b1 = get_bigrams(words1)
    b2 = get_bigrams(words2)
    if not b1 or not b2:
        return 0.0
    return len(b1.intersection(b2)) / len(b1.union(b2))

In [3]:
print("Generating Distractor A (Swapped)...")

distractor_a_comments =[]
length_dict = df.groupby('word_count')['original_comment'].apply(list).to_dict()

for index, row in df.iterrows():
    orig_comment = str(row['original_comment'])
    target_length = row['word_count']
    
    # Pool of comments with similar length (+/- 2 words)
    candidate_pool =[]
    for length in range(max(1, target_length - 2), target_length + 3):
        if length in length_dict:
            candidate_pool.extend(length_dict[length])
            
    candidate_pool = [c for c in candidate_pool if c != orig_comment]
    random.shuffle(candidate_pool)
    
    found = False
    for candidate in candidate_pool:
        # Mathematical proof of topic distortion
        if word_jaccard(orig_comment, candidate) < 0.3:
            distractor_a_comments.append(candidate)
            found = True
            break
            
    # Fallback to random selection if length constraints fail
    if not found:
        while True:
            rand_comment = df['original_comment'].sample(n=1).iloc[0]
            if word_jaccard(orig_comment, rand_comment) < 0.3:
                distractor_a_comments.append(rand_comment)
                break

df['distractor_a'] = distractor_a_comments
print("Distractor A generation complete.")

Generating Distractor A (Swapped)...
Distractor A generation complete.


In [4]:
print("Generating Distractor B (Shuffled)...")

distractor_b_comments =[]

for index, row in df.iterrows():
    words = str(row['original_comment']).lower().split()
    
    # Sentences with 1 or 2 words cannot be significantly shuffled
    if len(words) <= 2:
        words.reverse()
        distractor_b_comments.append(" ".join(words))
        continue
        
    best_shuffle = words[:]
    
    # Try up to 15 times to find a shuffle that breaks the syntax
    for _ in range(15):
        random.shuffle(best_shuffle)
        # Mathematical proof of syntactic distortion
        if bigram_jaccard(words, best_shuffle) < 0.1:
            break
            
    distractor_b_comments.append(" ".join(best_shuffle))

df['distractor_b'] = distractor_b_comments
print("Distractor B generation complete.")

Generating Distractor B (Shuffled)...
Distractor B generation complete.


In [5]:
# 1. Original 12k (Label 1)
df_orig = df.drop(columns=['distractor_a', 'distractor_b', 'word_count']).copy()
df_orig.rename(columns={'original_comment': 'comment'}, inplace=True)
df_orig['label'] = 1
df_orig.to_csv(os.path.join(PROCESSED_DIR, "01_Original_12k.csv"), index=False)

# 2. Distractor A 12k (Label 0)
df_dist_a = df.drop(columns=['original_comment', 'distractor_b', 'word_count']).copy()
df_dist_a.rename(columns={'distractor_a': 'comment'}, inplace=True)
df_dist_a['label'] = 0
df_dist_a.to_csv(os.path.join(PROCESSED_DIR, "02_Distractor_A_Swapped_12k.csv"), index=False)

# 3. Distractor B 12k (Label 0)
df_dist_b = df.drop(columns=['original_comment', 'distractor_a', 'word_count']).copy()
df_dist_b.rename(columns={'distractor_b': 'comment'}, inplace=True)
df_dist_b['label'] = 0
df_dist_b.to_csv(os.path.join(PROCESSED_DIR, "03_Distractor_B_Shuffled_12k.csv"), index=False)

print("Saved all 3 datasets separately.")

Saved all 3 datasets separately.
